In [1]:
# Linear Regression exercises using Cats dataset from R MASS library (seen from UIUC Stat 425 Lecture 1.1)
# Goal: predict Hwt (heart weight) from Bwt (body weight)
# File created: 25 November 2026
# Last edited: 25 November 2026

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

np.set_printoptions(suppress = True)
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("Datasets/Cats.csv")
data.head()

,rownames,Sex,Bwt,Hwt
0,1,F,2.0,7.0
1,2,F,2.0,7.4
2,3,F,2.0,9.5
3,4,F,2.1,7.2
4,5,F,2.1,7.3


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   rownames  144 non-null    int64  
 1   Sex       144 non-null    object 
 2   Bwt       144 non-null    float64
 3   Hwt       144 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 4.6+ KB


In [5]:
data.columns

Index(['rownames', 'Sex', 'Bwt', 'Hwt'], dtype='object')

In [6]:
# Predicting Hwt from Bwt using sklearn.linear_model's LinearRegression - R MASS library and UIUC Stat 425 Lecture 1.1 slide 17 of 24
print("Simple Linear Regression: Hwt = b0 + b1 x Bwt + eps")
Y = data["Hwt"].to_numpy()
X = data["Bwt"].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = LinearRegression()
model.fit(X, Y)
beta0 = model.intercept_
beta1 = model.coef_.item()
print("beta0 = %s, beta1 = %s" % (beta0, beta1))
Rsquare = model.score(X, Y)
print("Rsquare = %s" % Rsquare)

Simple Linear Regression: Hwt = b0 + b1 x Bwt + eps
(144, 1) (144,)
beta0 = -0.3566624328849457, beta1 = 4.034062698458522
Rsquare = 0.6466209132388888


In [7]:
class Jie_SimpleLinearRegression:
    
    def __init__(self):
        self.beta0 = None
        self.beta1 = None
        self.sigma2 = None
        self.Rsquare = None
        self.beta0_stdev = None
        self.beta1_stdev = None
        self.beta0_t = None
        self.beta1_t = None
        self.beta0_p = None
        self.beta1_p = None
        self.results = None
    
    def fit(self, X, Y, predictor_name):
        
        Xc = np.insert(X, 0, np.ones(len(X)), axis = 1)
        self.beta0, self.beta1 = np.linalg.inv(Xc.T @ Xc) @ Xc.T @ Y
        
        Yhat = self.predict(X)
        self.sigma2 = (np.linalg.norm(Y - Yhat) ** 2) / (len(Y) - 2)
        
        beta_var = np.diag(self.sigma2 * np.linalg.inv(Xc.T @ Xc))
        self.beta0_stdev, self.beta1_stdev = np.sqrt(beta_var)

        self.beta0_t = self.beta0 / self.beta0_stdev
        self.beta1_t = self.beta1 / self.beta1_stdev
        
        self.beta0_p = 2 * norm.sf(np.abs(self.beta0_t))
        self.beta1_p = 2 * norm.sf(np.abs(self.beta1_t))

        beta = np.array([self.beta0, self.beta1])
        beta_stdev = np.array([self.beta0_stdev, self.beta1_stdev])
        beta_t = np.array([self.beta0_t, self.beta1_t])
        beta_p = np.array([self.beta0_p, self.beta1_p])
        self.results = pd.DataFrame(zip(beta, beta_stdev, beta_t, beta_p),
                                    columns = ["beta", "beta stdev", "beta t", "beta p"],
                                    index = ["intercept", predictor_name])
        print(self.results)
        
    def score(self, X, Y):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = self.predict(X)
        RSS = np.linalg.norm(Y - Yhat) ** 2
        TSS = np.linalg.norm(Y - np.mean(Y)) ** 2
        self.Rsquare = 1 - RSS/TSS
        print("Rsquare = %s" % self.Rsquare)
        
    def predict(self, X):
        if (self.beta0 is None or self.beta1 is None):
            print("Model untrained/not fitted!")
            return None
        Yhat = (self.beta0 + self.beta1 * X).reshape(-1)
        return Yhat

In [8]:
# Predicting Hwt from Bwt by hand - R MASS library and UIUC Stat 425 Lecture 1.1 slide 17 of 24
print("Simple Linear Regression: Hwt = b0 + b1 x Bwt + eps")
Y = data["Hwt"].to_numpy()
predictor_name = "Bwt"
X = data[predictor_name].to_numpy().reshape((-1, 1))
print(X.shape, Y.shape)
model = Jie_SimpleLinearRegression()
model.fit(X, Y, predictor_name)
model.score(X, Y)

Simple Linear Regression: Hwt = b0 + b1 x Bwt + eps
(144, 1) (144,)
               beta  beta stdev     beta t        beta p
intercept -0.356662    0.692277  -0.515202  6.064120e-01
Bwt        4.034063    0.250261  16.119391  1.864382e-58
Rsquare = 0.6466209132388887
